# 14 | Five-seed curve averaging against notebook 11's 6214

**Purpose.** Freeze the protocol "same recipe, five seeds, average the 96-point curves, then `optimize_day`." The comparison is notebook 11's failure case, v2 Transformer **6214.36** / capture **0.690**, not the oracle near 9001.

**Headline locked in advance:** `listwise_transformer_mean`. This is the listwise Transformer selected by 30-day mean daily capture on track B (`tau_tgt=0.05`, the sharpest point on that grid, a **sharper window CE than notebook 12**). It was not chosen because it won on the 59 days. The other two bags (SPO+ LSTM, notebook-12 window-CE LSTM), the window vote, and the 50/50 mix are controls. A higher 59-day score does not change the headline.

**The standard error of a 59-day mean is about 550**, which does not resolve gaps of ±200–400. The three bags are **not ranked**. Phase B frozen numbers and the v1/v2 RMSE/MAE columns stay unchanged.

The 59-day window was already used in earlier notebooks. It is not an untouched holdout. Scores are raw dispatch scores, not currency.

Stored outputs were removed from this public copy. The executed charts used Chinese labels and cannot be redrawn without the raw series. Numbers in the text are the recorded results.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'phase_b').is_dir():
            return candidate
    raise FileNotFoundError('start this notebook from the repository root')


def show_figure(path: Path) -> None:
    display(Image(filename=str(path)))


REPO_ROOT = find_repo_root()
os.environ.setdefault('MPLCONFIGDIR', str(REPO_ROOT / '.matplotlib_cache'))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES = REPO_ROOT / 'outputs' / 'notebooks' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
REPORT = REPO_ROOT / 'reports' / 'holdout_seed_bag_v1'
V2 = REPO_ROOT / 'reports' / 'holdout_rmse_v2'
NB12 = REPO_ROOT / 'reports' / 'holdout_dispatch_loss_v1'
NB13 = REPO_ROOT / 'reports' / 'holdout_dfl_bakeoff_v1'
CLAUDE = REPO_ROOT / 'reports' / 'holdout_dfl_ltr_v1'
PHASE_B = REPO_ROOT / 'reports' / 'phase_b' / 'final_metrics.json'
MONEY_LINE = 6214.361782
HOLD_SE = 550.0
HEADLINE = 'listwise_transformer_mean'
assert PHASE_B.exists(), 'phase_b frozen metrics must still be present'
assert (REPORT / 'leaderboard.csv').exists(), 'run python -m experiments.holdout_seed_bag_v1.run first'
print({'repo_root': str(REPO_ROOT), 'report': str(REPORT.relative_to(REPO_ROOT))})


In [ ]:
manifest = json.loads((REPORT / 'manifest.json').read_text(encoding='utf-8'))
board = pd.read_csv(REPORT / 'leaderboard.csv')
daily = pd.read_csv(REPORT / 'dispatch_daily.csv')
daily['date'] = pd.to_datetime(daily['date'], format='mixed').dt.normalize()
assert manifest['holdout_not_used_for_selection'] is True
assert manifest['headline_bag'] == 'listwise_transformer'
assert manifest['money_line_source'].startswith('nb11')

v2_daily = pd.read_csv(V2 / 'dispatch_daily.csv')
v2_daily['date'] = pd.to_datetime(v2_daily['date'], format='mixed').dt.normalize()
nb11 = v2_daily.loc[v2_daily['model'] == 'transformer_contextual', 'realized_profit']
nb11_mean = float(nb11.mean())
assert abs(nb11_mean - MONEY_LINE) < 1e-3

nb12_board = pd.read_csv(NB12 / 'leaderboard.csv')
lstm12 = float(nb12_board.loc[nb12_board['model'] == 'lstm_dispatch', 'realized_profit_mean'].iloc[0])
spo13 = float('nan')
if (NB13 / 'leaderboard.csv').exists():
    b13 = pd.read_csv(NB13 / 'leaderboard.csv')
    if 'lstm_spoplus' in set(b13['model']):
        spo13 = float(b13.loc[b13['model'] == 'lstm_spoplus', 'realized_profit_mean'].iloc[0])

head = board.loc[board['model'] == HEADLINE].iloc[0]
print(board.loc[board['combiner'].isin(['curve_mean', 'curve_mean_two_bags', 'window_vote']),
                ['model', 'combiner', 'headline', 'realized_profit_mean', 'capture', 'seed_realized_min', 'seed_realized_max']
               ].to_string(index=False))


In [ ]:
def se_mean(series: pd.Series) -> float:
    s = pd.to_numeric(series, errors='coerce').dropna()
    return float(s.std(ddof=1) / np.sqrt(len(s)))

head_days = daily.loc[daily['model'] == HEADLINE].sort_values('date')
nb11_days = v2_daily.loc[v2_daily['model'] == 'transformer_contextual'].sort_values('date')
aligned = pd.merge(
    nb11_days[['date', 'realized_profit']].rename(columns={'realized_profit': 'nb11'}),
    head_days[['date', 'realized_profit']].rename(columns={'realized_profit': 'head'}),
    on='date',
)
delta = aligned['head'] - aligned['nb11']
without_1230 = aligned.loc[aligned['date'] != pd.Timestamp('2025-12-30')]
d1230 = aligned.loc[aligned['date'] == pd.Timestamp('2025-12-30')]

head_realized = float(head['realized_profit_mean'])
passed = head_realized > MONEY_LINE
lift = head_realized - MONEY_LINE
emp_se = se_mean(head_days['realized_profit'])

bag_means = board.loc[board['combiner'] == 'curve_mean'].copy()
spread = float(bag_means['realized_profit_mean'].max() - bag_means['realized_profit_mean'].min())

lines = []
if passed:
    lines.append(
        f"**Clears the line.** The pre-specified headline `{HEADLINE}` has 59-day realized score **{head_realized:.0f}**, "
        f"**{lift:+.0f}** versus notebook 11's 6214 (capture {float(head['capture']):.3f})."
    )
else:
    lines.append(
        f"**Does not clear the line.** Headline `{HEADLINE}` realized score **{head_realized:.0f}**; the line is 6214."
    )
lines.append(
    f"This is a **sharper notebook-12 window CE** (`tau_tgt=0.05`) times a Transformer times a 5-seed **curve average**. It is not a new-loss champion and it was not picked as the 59-day winner."
)
lines.append(
    f"Empirical SE of the headline daily series is **{emp_se:.0f}**. The plan used about **{HOLD_SE:.0f}**. The spread across the three curve-average bags is {spread:.0f}, still on the scale of the SE. **These 59 days do not say which bag is better.** They are not ranked, and the headline is not rewritten from the evaluation scores."
)
if len(d1230):
    lines.append(
        f"On 30 Dec the headline scores {float(d1230['head'].iloc[0]):.0f} versus notebook 11 at {float(d1230['nb11'].iloc[0]):.0f}. "
        f"Without that day the mean daily gap versus notebook 11 is {float((without_1230['head']-without_1230['nb11']).mean()):+.0f}."
    )
ce_bag = board.loc[board['model'] == 'window_ce_lstm_mean']
if len(ce_bag) and float(ce_bag['realized_profit_mean'].iloc[0]) > MONEY_LINE:
    lines.append(
        f"The window-CE LSTM bag realized **{float(ce_bag['realized_profit_mean'].iloc[0]):.0f}**, also above 6214. "
        "Averaging notebook 12 over five seeds already beats notebook 11, so the increase is not all due to the listwise loss."
    )
lines.append(
    f"Controls: notebook-12 single-seed LSTM {lstm12:.0f}; notebook-13 SPO+ LSTM {spo13:.0f}. "
    "These 59 days are not a new champion selection. Phase B and the v1/v2 RMSE columns are unchanged."
)
display(Markdown('### Result, computed from the saved outputs\n\n' + '\n\n'.join('- ' + s for s in lines)))


## Main table against 6214. The three bags are not ranked

RMSE is in the appendix only. The headline row is not rewritten if the 59-day score is higher or lower.


In [ ]:
rows = [
    {'recipe': 'nb11 v2 Transformer (best score)', 'definition': 'single seed, point RMSE', 'realized': nb11_mean, 'capture': nb11_mean / 9000.886096, 'vs 6214': 0.0, 'headline': False},
    {'recipe': 'nb12 lstm_dispatch', 'definition': 'single seed, window CE', 'realized': lstm12, 'capture': lstm12 / 9000.886096, 'vs 6214': lstm12 - MONEY_LINE, 'headline': False},
]
if np.isfinite(spo13):
    rows.append({'recipe': 'nb13 lstm_spoplus', 'definition': 'single seed, SPO+', 'realized': spo13, 'capture': spo13 / 9000.886096, 'vs 6214': spo13 - MONEY_LINE, 'headline': False})
for _, r in board.loc[board['combiner'] == 'curve_mean'].iterrows():
    rows.append({
        'recipe': r['model'],
        'definition': '5-seed curve average',
        'realized': r['realized_profit_mean'],
        'capture': r['capture'],
        'vs 6214': r['realized_profit_mean'] - MONEY_LINE,
        'headline': bool(r['headline']),
    })
mix = board.loc[board['model'] == 'listwise_spo_50_50']
if len(mix):
    r = mix.iloc[0]
    rows.append({'recipe': r['model'], 'definition': 'ablation: two bags, 50/50', 'realized': r['realized_profit_mean'], 'capture': r['capture'], 'vs 6214': r['realized_profit_mean'] - MONEY_LINE, 'headline': False})
TAB = pd.DataFrame(rows)
TAB['above_6214'] = TAB['realized'] > MONEY_LINE
display(TAB.style.format({'realized': '{:.1f}', 'capture': '{:.3f}', 'vs 6214': '{:+.1f}'}))

spread_tab = board.loc[board['combiner'] == 'curve_mean', ['model', 'seed_realized_min', 'seed_realized_max', 'realized_profit_mean']].copy()
spread_tab = spread_tab.rename(columns={'realized_profit_mean': 'curve_average_then_dispatch', 'seed_realized_min': 'single_seed_min', 'seed_realized_max': 'single_seed_max'})
display(Markdown('Within-bag min / max / average. The average is of curves, not of daily scores:'))
display(spread_tab.style.format('{:.1f}', subset=['single_seed_min', 'single_seed_max', 'curve_average_then_dispatch']))


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.2))
plot = TAB.sort_values('realized')
colors = ['#2a6f97' if h else '#adb5bd' for h in plot['headline']]
ax.barh(plot['recipe'], plot['realized'], color=colors)
ax.axvline(MONEY_LINE, color='#c1121f', ls='--', lw=1.2, label='nb11 6214')
ax.axvline(9000.886, color='#6c757d', ls=':', lw=1, label='oracle 9001, far above these scores')
ax.set_xlabel('59-day mean realized score')
ax.set_title('The headline is the pre-specified listwise Transformer bag, not the highest 59-day score')
ax.legend(loc='lower right', fontsize=8)
fig.tight_layout()
p1 = FIGURES / 'nb14_vs_6214.png'
fig.savefig(p1, dpi=140)
plt.close()
print(p1.relative_to(REPO_ROOT))
show_figure(p1)


In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 3.8))
labels, mins, maxs, means = [], [], [], []
for _, r in board.loc[board['combiner'] == 'curve_mean'].iterrows():
    labels.append(r['model'].replace('_mean', ''))
    mins.append(r['seed_realized_min'])
    maxs.append(r['seed_realized_max'])
    means.append(r['realized_profit_mean'])
x = np.arange(len(labels))
ax.vlines(x, mins, maxs, color='#6c757d', lw=3, label='single-seed min-max')
ax.scatter(x, means, color='#2a6f97', zorder=3, label='average curves, then dispatch')
ax.axhline(MONEY_LINE, color='#c1121f', ls='--', lw=1, label='6214')
ax.set_xticks(x, labels, rotation=15, ha='right')
ax.set_ylabel('59-day realized score')
ax.set_title('Seed range inside a bag. The averaged curve can score above the best single seed')
ax.legend(fontsize=8)
fig.tight_layout()
p2 = FIGURES / 'nb14_seed_span.png'
fig.savefig(p2, dpi=140)
plt.close()
print(p2.relative_to(REPO_ROOT))
show_figure(p2)


In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 3.6))
cmp = aligned.sort_values('date')
ax.plot(cmp['date'], cmp['nb11'], label='nb11 v2 Transformer', color='#adb5bd', lw=1.2)
ax.plot(cmp['date'], cmp['head'], label='headline bag', color='#2a6f97', lw=1.4)
if (cmp['date'] == pd.Timestamp('2025-12-30')).any():
    ax.axvline(pd.Timestamp('2025-12-30'), color='#c1121f', ls=':', lw=1, label='12-30')
ax.set_ylabel('that day\'s realized score')
ax.set_title('Daily gap versus notebook 11. 30 Dec is on the same scale as SE about 550')
ax.legend(fontsize=8)
fig.tight_layout()
p3 = FIGURES / 'nb14_daily_vs_11.png'
fig.savefig(p3, dpi=140)
plt.close()
print(p3.relative_to(REPO_ROOT))
show_figure(p3)
display(Markdown(
    f"Paired mean daily gap {float(delta.mean()):+.1f}; empirical SE {float(delta.std(ddof=1)/np.sqrt(len(delta))):.0f}. "
    f"Without 30 Dec, {float((without_1230['head']-without_1230['nb11']).mean()):+.1f}."
))


## Appendix: window-vote ablation and RMSE

A higher 59-day score for the window vote does not replace the headline. RMSE is not the pass criterion.


In [ ]:
app = board.loc[board['combiner'].isin(['curve_mean', 'window_vote', 'curve_mean_two_bags']),
                ['model', 'combiner', 'headline', 'realized_profit_mean', 'capture', 'RMSE', 'pred_profit_mean']].copy()
display(app.style.format({'realized_profit_mean': '{:.1f}', 'capture': '{:.3f}', 'RMSE': '{:.3f}', 'pred_profit_mean': '{:.1f}'}))
print('manifest seconds', manifest.get('seconds_total'), 'phase_b exists', PHASE_B.exists())
print('headline beats 6214', bool(head['beats_nb11_6214']))
